# Addis Ababa Road Traffic Accident Analysis
## Stage 4: Exploratory Data Analysis & Visualizations (ALX Module 1)

**Project:** Understanding and Trusting Road Accident Data: An Exploratory Analysis of Addis Ababa  
**Author:** YonInsights (`yo2120018@gmail.com`)  
**Kernel:** Python (Addis Ababa Road Safety)

---

## Step 1: Bar Charts & Pie Charts with Matplotlib
**Research Question:** How are recorded road traffic collisions distributed across accident severity categories in Addis Ababa?

- **Data required:** Categorical target variable `Accident_severity`.
- **Why Bar Chart:** Best for comparing absolute volumes along a common baseline without perceptual distortion.
- **Why Donut/Pie Chart:** Useful for illustrating relative composition of the whole (proportions), but small slices (e.g. Fatal injury at 2.4%) require explicit percentage callouts.
- **Figure output:** Saved to `../reports/figures/01_severity_distribution.png`.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# 1. Setup paths
DATA_PATH = Path("../Data/Addis_Ababa_city_RTA.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("Data/Addis_Ababa_city_RTA.csv")

FIG_DIR = Path("../reports/figures")
if not FIG_DIR.exists():
    FIG_DIR = Path("reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 2. Load dataset
df = pd.read_csv(DATA_PATH)

# 3. Compute counts and percentages for Accident_severity
severity_counts = df['Accident_severity'].value_counts()
categories = severity_counts.index.tolist()
counts = severity_counts.values
percentages = (counts / counts.sum()) * 100

# 4. Define palette: Muted Blue (Slight), Amber (Serious), Crimson (Fatal)
colors = ['#2b5c8f', '#e67e22', '#c0392b']

# 5. Create side-by-side comparison figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('Figure 1: Distribution of Recorded Accident Severity in Addis Ababa', fontsize=14, fontweight='bold', y=1.02)

# Subplot 1: Bar Chart (Absolute Counts)
bars = ax1.bar(categories, counts, color=colors, edgecolor='black', linewidth=0.8, width=0.55)
ax1.set_title('Absolute Incident Counts', fontsize=12, pad=10)
ax1.set_ylabel('Number of Recorded Collisions', fontsize=11)
ax1.set_xlabel('Assessed Severity Level', fontsize=11)
ax1.grid(axis='y', linestyle='--', alpha=0.5)

for bar, count, pct in zip(bars, counts, percentages):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 150,
             f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9.5, fontweight='bold')
ax1.set_ylim(0, max(counts) * 1.15)

# Subplot 2: Donut / Pie Chart (Proportions)
explode = (0.02, 0.05, 0.12)  # Emphasize the fatal category slice
wedges, texts, autotexts = ax2.pie(
    counts,
    labels=categories,
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    explode=explode,
    wedgeprops=dict(width=0.6, edgecolor='black', linewidth=0.8),
    pctdistance=0.75
)
for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')
    autotext.set_color('white')

ax2.set_title('Relative Proportions (%)', fontsize=12, pad=10)

plt.tight_layout()
save_path = FIG_DIR / "01_severity_distribution.png"
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"Figure saved to: {save_path}")
plt.show()

Figure saved to: ..\reports\figures\01_severity_distribution.png


---

## Step 2: DataFrame Preparation for Temporal Visualization
**Research Question:** At what hours of the day do recorded collisions peak in Addis Ababa, and does the proportion of severe or fatal injuries increase during nighttime hours?

### Data Preparation Rationale:
1. The raw `Time` column is formatted as strings (`HH:MM:SS`). We parse this column and extract the **`Hour` of the day** (`0` to `23`) using `pd.to_datetime().dt.hour`.
2. We cross-tabulate `Hour` against `Accident_severity` using `pd.crosstab()`.
3. We calculate the combined **Severe/Fatal Injury Rate (%)** for each hour: $$\text{Severe/Fatal Rate} = \frac{\text{Serious Injury} + \text{Fatal Injury}}{\text{Total Collisions at that Hour}} \times 100$$

In [2]:
# 1. Feature Engineering: Extract integer Hour (0-23) from Time string
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour

# 2. Compute hourly collision totals
hourly_counts = df.groupby('Hour').size()

# 3. Compute cross-tabulation between Hour and Severity
crosstab_hourly = pd.crosstab(df['Hour'], df['Accident_severity'])

# 4. Calculate Severe & Fatal combined rate
severe_fatal_pct = ((crosstab_hourly['Serious Injury'] + crosstab_hourly['Fatal injury']) / hourly_counts * 100).round(2)

# 5. Build structured analytical DataFrame for plotting
hourly_df = pd.DataFrame({
    'Total_Collisions': hourly_counts,
    'Slight_Injury_Count': crosstab_hourly['Slight Injury'],
    'Serious_Injury_Count': crosstab_hourly['Serious Injury'],
    'Fatal_Injury_Count': crosstab_hourly['Fatal injury'],
    'Severe_Fatal_Rate_Pct': severe_fatal_pct
})

print("=== Prepared Hourly Aggregation DataFrame (Sample Hours) ===")
hourly_df.loc[[0, 3, 7, 12, 17, 21]]

=== Prepared Hourly Aggregation DataFrame (Sample Hours) ===


---

## Step 3: Interactive Visualization with Plotly

### Why Plotly for this Question?
- **Dual-Axis Integration:** We need to plot two completely different scales simultaneously: **Volume** (bars ranging from 100 to 1,300 crashes) and **Rate** (line ranging from 10% to 25%).
- **Interactivity:** Road safety officials need to hover over specific hours to inspect exact counts and fatality rates.
- **Reproducible Export:** Saved as an optimized standalone HTML visualization to `../reports/figures/02_hourly_severity_interactive.html`.

In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Initialize figure with dual y-axes
fig_hourly = make_subplots(specs=[[{"secondary_y": True}]])

# 2. Trace 1: Hourly Collision Volume (Bar Chart on Primary Y-Axis)
fig_hourly.add_trace(
    go.Bar(
        x=hourly_df.index,
        y=hourly_df['Total_Collisions'],
        name='Collision Volume',
        marker_color='#2b5c8f',
        opacity=0.85,
        hovertemplate='<b>Hour %{x}:00</b><br>Total Crashes: %{y:,}<extra></extra>'
    ),
    secondary_y=False
)

# 3. Trace 2: Severe/Fatal Injury Rate (Line + Markers on Secondary Y-Axis)
fig_hourly.add_trace(
    go.Scatter(
        x=hourly_df.index,
        y=hourly_df['Severe_Fatal_Rate_Pct'],
        name='Severe/Fatal Rate (%)',
        mode='lines+markers',
        line=dict(color='#c0392b', width=3),
        marker=dict(size=7, color='#c0392b'),
        hovertemplate='<b>Hour %{x}:00</b><br>Severe/Fatal Rate: %{y:.1f}%<extra></extra>'
    ),
    secondary_y=True
)

# 4. Layout and formatting
fig_hourly.update_layout(
    title='<b>Figure 2: Hourly Collision Volume vs. Severe Injury Rate in Addis Ababa</b>',
    xaxis=dict(title='Hour of the Day (24-Hour Clock)', tickmode='linear', tick0=0, dtick=1),
    yaxis=dict(title='Total Recorded Collisions', showgrid=True),
    yaxis2=dict(title='Severe or Fatal Injury Rate (%)', showgrid=False, range=[0, 30]),
    hovermode='x unified',
    template='plotly_white',
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)'),
    margin=dict(l=60, r=60, t=70, b=60)
)

# 5. Export interactive figure
interactive_out = FIG_DIR / "02_hourly_severity_interactive.html"
fig_hourly.write_html(str(interactive_out), include_plotlyjs='cdn')
print(f"Interactive Plotly figure exported to: {interactive_out.resolve()}")

# Display inline in Jupyter notebook
fig_hourly.show()

Interactive Plotly figure exported to: D:\Projects\Addis-Ababa-Road-Safety-Analysis\reports\figures\02_hourly_severity_interactive.html


---

## Step 4: AI-Assisted Chart Generation and Verification

### The Engineering Principle:
AI is an excellent accelerator for drafting plotting boilerplate, but an analyst must **always verify AI-generated visualizations** against the physical dataset.

Common AI Hallucinations in Tabular Road Safety Data:
1. **Assuming non-existent columns:** Prompting an AI to "plot speeding accidents" might cause it to invent a non-existent `Speed_limit` column.
2. **Overplotting Blindness:** Prompting an AI for a scatter plot between discrete integer features (`Number_of_vehicles_involved` vs `Number_of_casualties`) often produces a plot where 13,000 points overlap completely, hiding data density.
3. **Verification Protocol:** Always check column existence, compute summary metrics with Pandas first, and audit the chart axes and legends.

---

## Step 5: Line Graphs and Scatter Plots

### Part A: Line Graph — Day of Week Progression
**Research Question:** How do crash volumes and severe outcomes progress from Monday to Sunday?
- **Data required:** `Day_of_week` ordered chronologically (Monday to Sunday) and `Accident_severity` counts.
- **Why Line Graph:** A line graph is suitable here because the days of the week form an ordered cyclical progression.

In [4]:
# 1. Define chronological day-of-week order
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# 2. Cross-tabulate and reindex
crosstab_dow = pd.crosstab(df['Day_of_week'], df['Accident_severity']).reindex(days_order)
crosstab_dow['Total'] = crosstab_dow.sum(axis=1)

# 3. Plot line graph with Matplotlib
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Total Volume
ax1.plot(days_order, crosstab_dow['Total'], marker='o', linewidth=2.5, color='#2b5c8f', label='Total Collisions')
ax1.set_title('Weekly Collision Volume Progression', fontsize=12, fontweight='bold', pad=10)
ax1.set_ylabel('Total Recorded Collisions', fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.set_ylim(1400, 2300)
for i, txt in enumerate(crosstab_dow['Total']):
    ax1.annotate(f'{txt:,}', (days_order[i], txt + 35), ha='center', fontsize=9.5, fontweight='bold')

# Subplot 2: Severity Progression
ax2.plot(days_order, crosstab_dow['Serious Injury'], marker='s', linewidth=2, color='#e67e22', label='Serious Injury')
ax2.plot(days_order, crosstab_dow['Fatal injury'], marker='^', linewidth=2, color='#c0392b', label='Fatal Injury')
ax2.set_title('Severe & Fatal Injury Progression', fontsize=12, fontweight='bold', pad=10)
ax2.set_ylabel('Injury Count', fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(frameon=True)
for i, txt in enumerate(crosstab_dow['Fatal injury']):
    ax2.annotate(f'{txt}', (days_order[i], txt + 3), ha='center', fontsize=9, fontweight='bold', color='#c0392b')

fig.suptitle('Figure 3: Weekly Progression of Collisions and Severity in Addis Ababa', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fig3_path = FIG_DIR / "03_day_of_week_trends.png"
plt.savefig(fig3_path, dpi=300, bbox_inches='tight')
print(f"Figure 3 saved to: {fig3_path}")
plt.show()

Figure 3 saved to: ..\reports\figures\03_day_of_week_trends.png


### Part B: Scatter Plot — Number of Vehicles vs. Number of Casualties
**Research Question:** Does an increase in vehicles involved in a collision linearly lead to more casualties?

- **Data required:** `Number_of_vehicles_involved` and `Number_of_casualties`.
- **The Overplotting Challenge:** Both features are small integers (1-7 vehicles, 1-8 casualties). Plotting 13,063 discrete integer points directly causes complete visual overlap.
- **The Technique:** Adding small random **jitter** ($$X \pm 0.18$$, $$Y \pm 0.18$$) with low alpha transparency ($$\alpha = 0.15$$) reveals the true density distribution.

In [5]:
import numpy as np

# 1. Add controlled jitter to disperse overlapping discrete points
np.random.seed(42)
jitter_x = df['Number_of_vehicles_involved'] + np.random.uniform(-0.18, 0.18, len(df))
jitter_y = df['Number_of_casualties'] + np.random.uniform(-0.18, 0.18, len(df))

# 2. Calculate linear regression trendline
m, b = np.polyfit(df['Number_of_vehicles_involved'], df['Number_of_casualties'], 1)
r_corr = df['Number_of_vehicles_involved'].corr(df['Number_of_casualties'])

# 3. Plot scatter plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(jitter_x, jitter_y, alpha=0.15, color='#006494', edgecolors='none', s=25)

# 4. Plot trendline
x_vals = np.array([1, 7])
ax.plot(x_vals, m * x_vals + b, color='#c0392b', linewidth=2, linestyle='--', label=f'Linear Trend (Slope: {m:.2f}, r = {r_corr:.2f})')

ax.set_title('Figure 4: Number of Vehicles Involved vs. Number of Casualties (with Jitter)', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Number of Vehicles Involved', fontsize=11)
ax.set_ylabel('Number of Casualties', fontsize=11)
ax.set_xticks(range(1, 8))
ax.set_yticks(range(1, 9))
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend(frameon=True, loc='upper left')

plt.tight_layout()
fig4_path = FIG_DIR / "04_vehicles_vs_casualties_scatter.png"
plt.savefig(fig4_path, dpi=300, bbox_inches='tight')
print(f"Figure 4 saved to: {fig4_path}")
plt.show()

Figure 4 saved to: ..\reports\figures\04_vehicles_vs_casualties_scatter.png


---

## Step 6: Advanced Visualizations with Seaborn and Matplotlib

### Visual 1: Heatmap — Day of Week vs. Hour Hotspots
**Research Question:** Where are the joint temporal collision hotspots in Addis Ababa?
- **Data required:** 2D cross-tabulation matrix: `Day_of_week` (rows) x `Hour` (columns).
- **Why Heatmap:** Colors represent collision intensity across a 7x24 grid, making commute peaks and weekend evening clusters instantly recognizable.

In [6]:
import seaborn as sns

# 1. Create 2D cross-tabulation matrix
dow_hour_matrix = pd.crosstab(df['Day_of_week'], df['Hour']).reindex(days_order)

# 2. Plot heatmap
fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(dow_hour_matrix, cmap='YlOrRd', annot=False, cbar_kws={'label': 'Collision Count'}, linewidths=0.5, ax=ax)
ax.set_title('Figure 5: Temporal Heatmap — Collision Hotspots by Day of Week and Hour', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Hour of Day (0 to 23)', fontsize=11)
ax.set_ylabel('Day of Week', fontsize=11)

plt.tight_layout()
fig5_path = FIG_DIR / "05_temporal_hotspot_heatmap.png"
plt.savefig(fig5_path, dpi=300, bbox_inches='tight')
print(f"Figure 5 saved to: {fig5_path}")
plt.show()

Figure 5 saved to: ..\reports\figures\05_temporal_hotspot_heatmap.png


### Visual 2: Box Plot — Casualties Across Vehicle Types
**Research Question:** Do heavy public transport vehicles (buses, lorries) show higher casualty counts per crash than passenger cars?
- **Data required:** Categorical `Type_of_vehicle` (top 6 classes) and continuous `Number_of_casualties`.
- **Why Box Plot:** Displays medians, quartiles (IQR), whiskers, and outliers, illustrating the dispersion of casualties across vehicle categories.

In [7]:
# 1. Filter for top 6 most frequent vehicle types
top_vehicles = df['Type_of_vehicle'].value_counts().head(6).index
df_top_veh = df[df['Type_of_vehicle'].isin(top_vehicles)]

# 2. Plot box plot
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.boxplot(
    data=df_top_veh, x='Type_of_vehicle', y='Number_of_casualties', hue='Type_of_vehicle',
    palette='Blues_r', ax=ax, showmeans=True, legend=False,
    meanprops={'marker': 'o', 'markerfacecolor': 'red', 'markeredgecolor': 'red'}
)
ax.set_title('Figure 6: Distribution of Casualties by Primary Vehicle Type (Top 6)', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Vehicle Type', fontsize=11)
ax.set_ylabel('Number of Casualties', fontsize=11)
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
fig6_path = FIG_DIR / "06_vehicle_type_casualties_boxplot.png"
plt.savefig(fig6_path, dpi=300, bbox_inches='tight')
print(f"Figure 6 saved to: {fig6_path}")
plt.show()

Figure 6 saved to: ..\reports\figures\06_vehicle_type_casualties_boxplot.png


### Visual 3: Violin Plot — Casualty Probability Density Across Severity Levels
**Research Question:** What is the full probability distribution and density shape of casualty numbers across the three severity outcomes?
- **Data required:** `Accident_severity` and `Number_of_casualties`.
- **Why Violin Plot:** While box plots only show summary points (median/IQR), violin plots estimate the continuous kernel density (KDE), showing exactly where the probability mass concentrates.

In [8]:
# 1. Plot violin plot with quartile markers
fig, ax = plt.subplots(figsize=(9, 5))
severity_order = ['Slight Injury', 'Serious Injury', 'Fatal injury']

sns.violinplot(
    data=df, x='Accident_severity', y='Number_of_casualties', order=severity_order, hue='Accident_severity',
    palette=['#2b5c8f', '#e67e22', '#c0392b'], ax=ax, inner='quartile', cut=0, legend=False
)
ax.set_title('Figure 7: Casualty Density and Distribution Across Accident Severity Levels', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Assessed Severity Level', fontsize=11)
ax.set_ylabel('Number of Casualties', fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
fig7_path = FIG_DIR / "07_severity_casualties_violinplot.png"
plt.savefig(fig7_path, dpi=300, bbox_inches='tight')
print(f"Figure 7 saved to: {fig7_path}")
plt.show()

Figure 7 saved to: ..\reports\figures\07_severity_casualties_violinplot.png


### Visual 4: Pair Plot — Multivariate Continuous Interactions
**Research Question:** What are the multidimensional correlations between vehicles involved, casualties, and time of day, stratified by accident severity?
- **Data required:** `Number_of_vehicles_involved`, `Number_of_casualties`, `Hour`, and `Accident_severity`.
- **Why Pair Plot:** Generates an N x N matrix of scatter plots and diagonal KDE distributions to reveal bivariate interactions across all numeric dimensions simultaneously.

In [9]:
# 1. Select numeric variables and target
num_df = df[['Number_of_vehicles_involved', 'Number_of_casualties', 'Hour', 'Accident_severity']].dropna()

# 2. Plot Seaborn pair plot
pair_grid = sns.pairplot(
    num_df, hue='Accident_severity', hue_order=severity_order,
    palette=['#2b5c8f', '#e67e22', '#c0392b'], diag_kind='kde', plot_kws={'alpha': 0.3, 's': 18}
)
pair_grid.fig.suptitle('Figure 8: Pair Plot — Multivariate Relationships Stratified by Severity', y=1.02, fontsize=12, fontweight='bold')

fig8_path = FIG_DIR / "08_multivariate_pairplot.png"
pair_grid.savefig(fig8_path, dpi=300, bbox_inches='tight')
print(f"Figure 8 saved to: {fig8_path}")
plt.show()

Figure 8 saved to: ..\reports\figures\08_multivariate_pairplot.png
